In [17]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from nuplan.planning.training.preprocessing.utils.feature_cache import FeatureCachePickle

from sledge.metrics.reconstruction.utils import (
    ReconstructionMetricConfig,
    node_poses,
    one_to_one_matching,
    convert_to_digraph,
    keep_nodes_in_frame,
)
from sledge.common.helper.cache_helper import find_feature_paths
from sledge.autoencoder.preprocessing.features.graph_feature import GraphConfig, Graph

In [21]:
SLEDGE_CACHE_ROOT = os.environ.get("SLEDGE_CACHE_ROOT")

metric_cache_path = Path(SLEDGE_CACHE_ROOT) / "metric_cache"
reconstruction_cache_path = Path(SLEDGE_CACHE_ROOT) / "reconstruction_cache"

In [22]:
def plot_graph(G, G_hat):
    fig, ax = plt.subplots()

    G_node2idx = {node: idx for idx, node in enumerate(G.nodes)}
    G_hat_node2idx = {node: idx for idx, node in enumerate(G_hat.nodes)}

    poses_a, poses_b = node_poses(G), node_poses(G_hat)
    matched_pairs_nodes = one_to_one_matching(G, G_hat, metric_config=ReconstructionMetricConfig())

    matched_pairs = np.array([[G_node2idx[i], G_hat_node2idx[j]] for i, j in matched_pairs_nodes])
    matched_a = np.zeros(len(poses_a), dtype=bool)
    matched_b = np.zeros(len(poses_b), dtype=bool)

    if matched_pairs.ndim == 2:
        matched_a[matched_pairs[:, 0]] = True
        matched_b[matched_pairs[:, 1]] = True
        offsets = poses_a[matched_pairs[:, 0], :2] - poses_b[matched_pairs[:, 1], :2]

        for offset, point in zip(offsets, poses_b[matched_pairs[:, 1], :2]):
            stack = np.vstack([point, point + offset]).T
            ax.plot(*stack, color="black", zorder=2)

    dot_size = 15

    for node_start, node_end in list(G_hat.edges()):
        start_idx, end_idx = G_hat_node2idx[node_start], G_hat_node2idx[node_end]
        stack = np.vstack([poses_b[start_idx, :2], poses_b[end_idx, :2]]).T
        ax.plot(stack[0], stack[1], color="lightgray", zorder=0)

    obj = ax.scatter(
        *poses_b[matched_b, :2].T,
        c=poses_b[matched_b, 2],
        s=dot_size,
        cmap="viridis",
        label="TP",
        zorder=1,
        vmax=np.pi,
        vmin=-np.pi,
    )
    ax.scatter(*poses_b[~matched_b, :2].T, s=25, color="C1", label="FP", zorder=1, marker="X")
    ax.scatter(*poses_a[~matched_a, :2].T, s=25, color="C3", label="FN", zorder=1, marker="P")

    obj.set_clim(-np.pi, np.pi)

    ax.axis("scaled")
    ax.set_xlim(-65 / 2, 65 / 2)
    ax.set_ylim(-65 / 2, 65 / 2)

    # ax.colorbar(location='left', label="angle [rad]")
    plt.colorbar(obj, ax=ax, location="left", label="angle [rad]")
    ax.legend(loc="lower left")

    fig.tight_layout()
    # fig.savefig(os.path.join(save_dir, f"{name}.png"), dpi=500, bbox_inches="tight", pad_inches=0.1)

    # plt.cla()
    # plt.close(fig)


def plot_single_graph(G_hat):
    fig, ax = plt.subplots()

    G_hat_node2idx = {node: idx for idx, node in enumerate(G_hat.nodes)}

    poses_b = node_poses(G_hat)

    matched_b = np.zeros(len(poses_b), dtype=bool)

    # for point in poses_b:
    #     stack = np.vstack([point, point + offset]).T
    #     ax.plot(*stack, color="black", zorder=2)

    dot_size = 15

    for node_start, node_end in list(G_hat.edges()):
        start_idx, end_idx = G_hat_node2idx[node_start], G_hat_node2idx[node_end]
        stack = np.vstack([poses_b[start_idx, :2], poses_b[end_idx, :2]]).T
        ax.plot(stack[0], stack[1], color="black", zorder=0)

    obj = ax.scatter(
        *poses_b[:, :2].T,
        c=poses_b[:, 2],
        s=dot_size,
        cmap="viridis",
        label="TP",
        zorder=1,
        vmax=np.pi,
        vmin=-np.pi,
    )

    obj.set_clim(-np.pi, np.pi)

    ax.axis("scaled")
    ax.set_xlim(-64 / 2, 64 / 2)
    ax.set_ylim(-64 / 2, 64 / 2)

    # ax.colorbar(location='left', label="angle [rad]")
    plt.colorbar(obj, ax=ax, location="left", label="angle [rad]")
    ax.legend(loc="lower left")
    plt.show()
    # fig.tight_layout()
    # fig.savefig(os.path.join(save_dir, f"{name}.png"), dpi=500, bbox_inches="tight", pad_inches=0.1)

    # plt.cla()
    # plt.close(fig)

In [25]:
# Load ground-truth and predicted feature paths



target_paths = find_feature_paths(metric_cache_path, "graph")
prediction_paths = find_feature_paths(reconstruction_cache_path, "autoencoder_graph")

# Convert to scenario token dictionary
target_path_dict = {path.parent.name: path for path in target_paths}
prediction_path_dict = {path.parent.name: path for path in prediction_paths}

# Sort dictionaries
target_path_dict = dict(sorted(target_path_dict.items()))
prediction_path_dict = dict(sorted(prediction_path_dict.items()))

assert target_path_dict.keys() == prediction_path_dict.keys()

storing_mechanism = FeatureCachePickle()
graph_config = GraphConfig(frame=[64, 64], pose_interval=1.5)



In [ ]:

idx = 13334
target_graph = storing_mechanism.load_computed_feature_from_folder(list(target_path_dict.values())[idx], Graph)
prediction_graph = storing_mechanism.load_computed_feature_from_folder(list(prediction_path_dict.values())[idx], Graph)

# convert to digraph
G_target = convert_to_digraph(target_graph)
G_prediction = convert_to_digraph(prediction_graph)

# remove invalid nodes
G_target = keep_nodes_in_frame(G_target, graph_config)
G_prediction = keep_nodes_in_frame(G_prediction, graph_config)

plot_graph(G_target, G_prediction)

In [ ]:
plot_single_graph(G_prediction)

In [ ]:
plot_single_graph(G_target)